## Rate Limiting and Exponential Backoff

### 1. What is rate limiting?

Rate limiting controls how many requests a client is allowed to make within a period of time.

For example, suppose an API says:

60 requests per minute

You cannot send 100 requests immediately. The API may reject requests after the limit is reached.

A simple example:

```
Allowed: 60 requests / minute

Request 1   ✓
Request 2   ✓
...
Request 60  ✓
Request 61  ✗  → HTTP 429 Too Many Requests
```

The purpose is to:

- Protect the server
- Prevent abuse
- Ensure fair usage
- Prevent overload
- Control infrastructure costs

### 2. Rate limiting vs exponential backoff

Think of them as two different mechanisms.

- Rate limiting = "How fast am I allowed to send requests?"
- Exponential backoff = "What should I do after a request fails?"

For example:

```
Client
  |
  | Request
  v
API
  |
  | 429 Too Many Requests
  v
Client
  |
  | wait 1 second
  | retry
  |
  | 429
  |
  | wait 2 seconds
  | retry
  |
  | 429
  |
  | wait 4 seconds
  | retry
```

So:

Rate limiter controls request frequency.

Backoff controls retry timing.

### 3. Why exponential backoff?

Suppose your application gets a 429.

A bad implementation would be:

```
while True:
    response = call_api()

    if response.status_code == 429:
        continue
```

This creates a retry storm:

```
Request → 429
Request → 429
Request → 429
Request → 429
Request → 429
...
```

You're actually making the server's problem worse.

Instead, use exponential backoff:

```
Attempt 1 → wait 1 sec
Attempt 2 → wait 2 sec
Attempt 3 → wait 4 sec
Attempt 4 → wait 8 sec
Attempt 5 → wait 16 sec
```

The basic formula is:
```
delay = base_delay^retry_attempt
```
For example:
```
base = 1 second


attempt 0 → 1 second
attempt 1 → 2 seconds
attempt 2 → 4 seconds
attempt 3 → 8 seconds
attempt 4 → 16 seconds
```

Usually you also add jitter:
```
delay = exponential_backoff + random_jitter
```
because otherwise thousands of clients might retry at exactly the same time.

### 4. How do you implement it?

Let's say an API allows:

5 requests/second

and occasionally returns 429.

A simple retry implementation could look like:

In [ ]:
import time
import requests

BASE_DELAY = 2

def fetch(url, max_retries=5):

    for attempt in range(max_retries):

        response = requests.get(url)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            delay = (BASE_DELAY ** attempt) + random.uniform(0, 1)  # adds random jitter

            print(f"Rate limited. Waiting {delay} seconds...")
            time.sleep(delay)

            continue

        # Other errors
        response.raise_for_status()

    raise Exception("Max retries exceeded")